In [276]:
import sys
from pathlib import Path

# Trova la radice sia dalla cartella notebook sia dalla radice del progetto.
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "helpers.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2

import pandas as pd
import src.helpers as H


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# CGM

In [277]:
cgm = pd.read_csv("/home/andrea/projects/TimestralGluco/data/tandemsource_data/merged/merged-CGM.csv")

In [278]:
cgm.head()

,Tipo di dispositivo,Numero di serie,Descrizione,Data e ora dell'evento,Letture (mg/dL)
0,⁦Pompa per insulina t:slim X2™⁩,1257006,Valore glucosio stimato,2025-11-28T13:55:14,219
1,⁦Pompa per insulina t:slim X2™⁩,1257006,Valore glucosio stimato,2025-11-28T14:00:13,212
2,⁦Pompa per insulina t:slim X2™⁩,1257006,Valore glucosio stimato,2025-11-28T14:05:13,216
3,⁦Pompa per insulina t:slim X2™⁩,1257006,Valore glucosio stimato,2025-11-28T14:10:13,222
4,⁦Pompa per insulina t:slim X2™⁩,1257006,Valore glucosio stimato,2025-11-28T14:15:13,221


In [279]:
cgm.drop(columns = ['Tipo di dispositivo','Numero di serie','Descrizione'],inplace = True)
cgm.head()

,Data e ora dell'evento,Letture (mg/dL)
0,2025-11-28T13:55:14,219
1,2025-11-28T14:00:13,212
2,2025-11-28T14:05:13,216
3,2025-11-28T14:10:13,222
4,2025-11-28T14:15:13,221


In [280]:
cgm.rename(columns={"Data e ora dell'evento": "datetime", "Letture (mg/dL)": "glucose"}, inplace=True)
cgm["datetime"] = pd.to_datetime(cgm["datetime"], format="%Y-%m-%dT%H:%M:%S")
cgm["glucose"] = pd.to_numeric(cgm["glucose"], errors="coerce")
cgm.head()

,datetime,glucose
0,2025-11-28 13:55:14,219
1,2025-11-28 14:00:13,212
2,2025-11-28 14:05:13,216
3,2025-11-28 14:10:13,222
4,2025-11-28 14:15:13,221


In [281]:
cgm.to_csv("/home/andrea/projects/TimestralGluco/data/cleaned/cleaned-CGM.csv", index=False)

# Bolo

In [282]:
bolo = pd.read_csv("/home/andrea/projects/TimestralGluco/data/tandemsource_data/merged/merged-Bolo.csv")
bolo = H.drop_uniform(bolo)
H.summarize_dataframe(bolo)

Colonna rimossa: Tipologia | Valore unico: 'Bolo'
Colonna rimossa: Numero di serie | Valore unico: np.int64(1257006)
                                 Colonna  Valori unici                                                  Esempi (max 3) Missing (%) Tipo dati
                            Tipo di bolo             5                     [Automatico, Alimento/Correzione, Alimenti]       0.00%       str
         Modalità di erogazione del bolo             3                              [Automatico, Standard, Prolungato]       0.00%       str
                        Glicemia (mg/dL)           244                                                 [258, 290, 267]       0.00%     int64
               Data/Ora di completamento          4566 [2025-11-28T14:33:52, 2025-11-28T15:06:13, 2025-11-28T16:14:21]       0.00%       str
                        Insulina erogata           303                                              [0.05, 1.49, 0.63]       0.00%   float64
                                   Pa

In [283]:
date_columns = [
    "Data/Ora di completamento",
    "Orario/Data d'inizio del bolo esteso",
    "Completamento del bolo esteso",
]
for column in date_columns:
    bolo[column] = pd.to_datetime(bolo[column], format="%Y-%m-%dT%H:%M:%S")

bolo.head()

,Tipo di bolo,Modalità di erogazione del bolo,Glicemia (mg/dL),Data/Ora di completamento,Insulina erogata,Pasto,Correzione erogata,Descrizione dello stato di completamento,Orario/Data d'inizio del bolo esteso,Completamento del bolo esteso,Insulina erogata con bolo esteso,Stato di completamento del bolo esteso,Percentuale standard,Durata (min),Quantità Carboidrati,Glicemia di riferimento (mg/dL),Fattore di correzione,Rapporto insulina:carboidrati
0,Automatico,Automatico,258,2025-11-28 14:33:52,0.05,0.00,0.00,Completato,NaT,NaT,NaN,NaN,100,0,0,110,115,0.0
1,Alimento/Correzione,Standard,290,2025-11-28 15:06:13,1.49,0.75,0.73,Completato,NaT,NaT,NaN,NaN,100,0,30,110,80,40.0
2,Automatico,Automatico,267,2025-11-28 16:14:21,0.63,0.00,0.00,Completato,NaT,NaT,NaN,NaN,100,0,0,110,80,0.0
3,Alimenti,Standard,280,2025-11-28 16:15:29,0.75,0.75,0.00,Completato,NaT,NaT,NaN,NaN,100,0,30,110,80,40.0
4,Automatico,Automatico,251,2025-11-28 17:19:01,0.47,0.00,0.00,Completato,NaT,NaT,NaN,NaN,100,0,0,110,85,0.0


In [284]:
bolo["Tipo di bolo"].value_counts()

Tipo di bolo
Automatico             2744
Alimento/Correzione     987
Alimenti                596
Correzione              228
Sovrascritto             11
Name: count, dtype: int64

In [285]:
# Riclassifica secondo le regole definite per questo dataset.
tipo_bolo_prima = bolo["Tipo di bolo"].copy()

# Zero carboidrati: Automatico, con priorità sulle altre condizioni.
zero_carboidrati = bolo["Quantità Carboidrati"].eq(0)
bolo.loc[zero_carboidrati, "Tipo di bolo"] = "Automatico"

# Unisci le categorie manuali rimanenti prima di applicare le condizioni.
# I boli Automatico restano separati.
tipi_da_riclassificare = [
    "Correzione", "Alimenti", "Sovrascritto", "Alimento/Correzione"
]
candidati = tipo_bolo_prima.isin(tipi_da_riclassificare) & ~zero_carboidrati
bolo.loc[candidati, "Tipo di bolo"] = "Alimento/Correzione"

# Fascia serale: dalle 19:30 alle 22:00 incluse.
orario_bolo = (
    bolo["Data/Ora di completamento"]
    - bolo["Data/Ora di completamento"].dt.normalize()
)
fascia_serale = orario_bolo.between(
    pd.Timedelta(hours=19, minutes=30),
    pd.Timedelta(hours=22),
    inclusive="both",
)
# In questa fascia è pasto se carboidrati > 41,
# oppure glicemia < 270 e carboidrati > 29,
# oppure glicemia < 210 e carboidrati > 21 (regola precedente).
pasto_serale = fascia_serale & (
    bolo["Quantità Carboidrati"].gt(41)
    | (
        bolo["Glicemia (mg/dL)"].lt(270)
        & bolo["Quantità Carboidrati"].gt(29)
    )
    | (
        bolo["Glicemia (mg/dL)"].lt(210)
        & bolo["Quantità Carboidrati"].gt(21)
    )
)

# Pranzo: dalle 12:30 alle 15:00 incluse, glicemia < 250 e carboidrati > 21.
fascia_pranzo = orario_bolo.between(
    pd.Timedelta(hours=12, minutes=30),
    pd.Timedelta(hours=15),
    inclusive="both",
)
pasto_pranzo = (
    fascia_pranzo
    & bolo["Glicemia (mg/dL)"].lt(250)
    & bolo["Quantità Carboidrati"].gt(21)
)

# Meno di 20 g fuori dalla fascia mattutina 06:00-11:30 inclusa.
fascia_mattina = orario_bolo.between(
    pd.Timedelta(hours=6),
    pd.Timedelta(hours=11, minutes=30),
    inclusive="both",
)
correzione_fuori_mattina = (
    bolo["Quantità Carboidrati"].lt(20)
    & orario_bolo.notna()
    & ~fascia_mattina
)

# Le condizioni di pasto hanno priorità sulle condizioni di correzione.
pasto_secondo_regole = (
    bolo["Quantità Carboidrati"].gt(50)
    | bolo["Glicemia (mg/dL)"].lt(180)
    | bolo["Modalità di erogazione del bolo"].eq("Prolungato")
    | pasto_serale
    | pasto_pranzo
)
ora_bolo = bolo["Data/Ora di completamento"].dt.hour
# Intervalli: [00:00, 05:00) e [16:00, 19:00).
fascia_correzione = ora_bolo.lt(5) | (ora_bolo.ge(16) & ora_bolo.lt(19))

# Vale a qualsiasi ora: glicemia strettamente > 250 e carboidrati < 35.
correzione_glicemia_alta = (
    bolo["Glicemia (mg/dL)"].gt(250)
    & bolo["Quantità Carboidrati"].lt(35)
)

# Ulteriore condizione valida a qualsiasi ora: glicemia > 200 e carboidrati < 20.
correzione_pochi_carboidrati = (
    bolo["Glicemia (mg/dL)"].gt(200)
    & bolo["Quantità Carboidrati"].lt(21)
)

# Ulteriore condizione valida a qualsiasi ora: glicemia > 270 e carboidrati < 41.
correzione_glicemia_molto_alta = (
    bolo["Glicemia (mg/dL)"].gt(270)
    & bolo["Quantità Carboidrati"].lt(41)
)

# Assegna Alimenti o Correzione quando le condizioni lo richiedono.
# I casi senza una condizione decisiva vengono risolti nel passaggio finale.
bolo.loc[
    candidati & pasto_secondo_regole,
    "Tipo di bolo",
] = "Alimenti"
bolo.loc[
    candidati
    & (
        fascia_correzione
        | correzione_glicemia_alta
        | correzione_pochi_carboidrati
        | correzione_glicemia_molto_alta
        | correzione_fuori_mattina
    )
    & ~pasto_secondo_regole,
    "Tipo di bolo",
] = "Correzione"

# Ultimo passaggio: risolvi solo i casi ancora in Alimento/Correzione.
rimanenti = bolo["Tipo di bolo"].eq("Alimento/Correzione")
bolo.loc[
    rimanenti & bolo["Quantità Carboidrati"].le(20), "Tipo di bolo"
] = "Correzione"
bolo.loc[
    rimanenti & bolo["Quantità Carboidrati"].gt(20), "Tipo di bolo"
] = "Alimenti"

modificati = bolo["Tipo di bolo"].ne(tipo_bolo_prima)
print(f"Boli riclassificati: {modificati.sum()}")
print(pd.crosstab(
    tipo_bolo_prima[modificati].rename("Prima"),
    bolo.loc[modificati, "Tipo di bolo"].rename("Dopo"),
).to_string())


Boli riclassificati: 1471
Dopo                 Alimenti  Automatico  Correzione
Prima                                                
Alimenti                    0           0         245
Alimento/Correzione       522           0         465
Correzione                  0         228           0
Sovrascritto                2           9           0


In [286]:
bolo["Tipo di bolo"].value_counts()


Tipo di bolo
Automatico    2981
Alimenti       875
Correzione     710
Name: count, dtype: int64

In [287]:
colonne_prioritarie = [
    "Tipo di bolo",
    "Modalità di erogazione del bolo",
    "Glicemia (mg/dL)",
    "Data/Ora di completamento",
    "Quantità Carboidrati",
    "Insulina erogata",
]
bolo = bolo[colonne_prioritarie + [
    column for column in bolo.columns if column not in colonne_prioritarie
]]


In [288]:
# Conserva solo le somministrazioni completate prima di eliminare le colonne.
bolo = bolo.loc[
    bolo["Descrizione dello stato di completamento"].eq("Completato")
].copy()

bolo.drop(columns=["Rapporto insulina:carboidrati", 
                   "Fattore di correzione",
                   "Glicemia di riferimento (mg/dL)",
                   "Stato di completamento del bolo esteso",
                   "Durata (min)",
                   "Percentuale standard",
                   "Completamento del bolo esteso",
                   "Orario/Data d'inizio del bolo esteso",
                   "Insulina erogata con bolo esteso",
                   "Descrizione dello stato di completamento"
                   ], inplace=True)

bolo.rename(columns={"Data/Ora di completamento": "datetime"}, inplace=True)
bolo

,Tipo di bolo,Modalità di erogazione del bolo,Glicemia (mg/dL),datetime,Quantità Carboidrati,Insulina erogata,Pasto,Correzione erogata
0,Automatico,Automatico,258,2025-11-28 14:33:52,0,0.05,0.00,0.00
1,Correzione,Standard,290,2025-11-28 15:06:13,30,1.49,0.75,0.73
2,Automatico,Automatico,267,2025-11-28 16:14:21,0,0.63,0.00,0.00
3,Correzione,Standard,280,2025-11-28 16:15:29,30,0.75,0.75,0.00
4,Automatico,Automatico,251,2025-11-28 17:19:01,0,0.47,0.00,0.00
...,...,...,...,...,...,...,...,...
4561,Automatico,Automatico,226,2026-10-22 14:35:31,0,0.17,0.00,0.00
4562,Automatico,Standard,253,2026-10-22 15:26:32,0,1.00,0.00,1.00
4563,Automatico,Standard,267,2026-10-22 15:54:33,0,0.70,0.00,0.70
4564,Automatico,Standard,284,2026-10-22 16:21:21,0,0.74,0.00,0.74


In [289]:
bolo.to_csv(
    "/home/andrea/projects/TimestralGluco/data/cleaned/cleaned-Bolo.csv",
    index=False,
)

# Dosi basali

In [290]:
db = pd.read_csv("/home/andrea/projects/TimestralGluco/data/tandemsource_data/merged/merged-Dosi basali.csv")

In [291]:
db = H.drop_uniform(db)

Colonna rimossa: Tipo di dispositivo | Valore unico: '\u2066Pompa per insulina t:slim X2™\u2069'
Colonna rimossa: Numero di serie | Valore unico: np.int64(1257006)


In [292]:
H.summarize_dataframe(db)

                                   Colonna  Valori unici                                                  Esempi (max 3) Missing (%) Tipo dati
                    Data e ora dell'evento         94414 [2025-11-28T13:58:23, 2025-11-28T14:03:22, 2025-11-28T14:08:23]       0.00%       str
Dose basale prescritta (unità di insulina)           144                                            [0.025, 0.04, 0.053]       0.00%   float64

Righe: 94415 | Colonne: 2


In [293]:
db.rename(columns={"Data e ora dell'evento": "datetime"}, inplace=True)
db["datetime"] = pd.to_datetime(db["datetime"], format="%Y-%m-%dT%H:%M:%S")
db

,datetime,Dose basale prescritta (unità di insulina)
0,2025-11-28 13:58:23,0.025
1,2025-11-28 14:03:22,0.025
2,2025-11-28 14:08:23,0.025
3,2025-11-28 14:13:23,0.025
4,2025-11-28 14:18:23,0.025
...,...,...
94410,2026-10-22 17:50:21,0.029
94411,2026-10-22 17:55:21,0.030
94412,2026-10-22 18:00:22,0.034
94413,2026-10-22 18:05:22,0.029


In [294]:
db.to_csv(
    "/home/andrea/projects/TimestralGluco/data/cleaned/cleaned-Basal.csv",
    index=False,
)

# Somministrazione oraria basale

In [295]:
sob = pd.read_csv("/home/andrea/projects/TimestralGluco/data/tandemsource_data/merged/merged-Somministrazione oraria basale.csv")
sob = H.drop_uniform(sob)
H.summarize_dataframe(sob)

Colonna rimossa: Numero di serie | Valore unico: np.int64(1257006)
         Colonna  Valori unici                       Esempi (max 3) Missing (%) Tipo dati
Data dell'evento           329 [2025-11-28, 2025-11-29, 2025-11-30]       0.00%       str
               0           245                [0.402, 0.716, 0.477]       2.10%   float64
               1           260                [0.596, 1.178, 0.575]       1.80%   float64
               2           254                 [0.91, 0.329, 0.539]       2.10%   float64
               3           271                [0.492, 0.222, 0.308]       1.80%   float64
               4           262                [0.628, 0.114, 0.188]       1.80%   float64
               5           255                [0.522, 0.186, 0.634]       1.80%   float64
               6           237                [0.525, 0.659, 0.401]       1.80%   float64
               7           224                [0.457, 0.362, 0.333]       1.80%   float64
               8           203   

In [296]:
sob

,Data dell'evento,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,2025-11-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.380,0.570,0.518,0.526,0.572,0.363,0.355,0.350,0.384,0.581
1,2025-11-29,0.402,0.596,0.910,0.492,0.628,0.522,0.525,0.457,0.473,...,0.300,0.806,0.824,0.715,0.501,0.717,0.389,0.391,0.350,0.450
2,2025-11-30,0.716,1.178,0.329,0.222,0.114,0.186,0.659,0.362,0.446,...,0.333,0.454,0.563,0.321,0.350,0.722,0.422,0.581,0.715,0.689
3,2025-12-01,0.477,0.575,0.539,0.308,0.188,0.634,0.401,0.333,0.516,...,0.532,0.865,0.738,0.425,0.444,0.080,0.254,0.541,0.450,0.580
4,2025-12-02,0.471,0.611,0.362,0.042,0.615,0.200,0.441,0.441,0.302,...,0.300,0.687,0.543,0.632,0.371,0.663,0.482,0.350,0.700,0.473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,2026-10-18,0.345,0.699,0.487,0.264,0.402,0.333,0.507,0.299,0.264,...,0.300,0.840,0.450,0.381,0.371,0.433,0.350,0.350,0.478,0.743
330,2026-10-19,0.430,1.029,0.420,0.240,0.355,0.381,0.414,0.315,0.357,...,0.275,0.450,0.467,0.422,0.350,0.321,0.320,0.414,0.393,0.369
331,2026-10-20,0.364,0.388,0.589,0.401,0.217,0.267,0.622,0.333,0.296,...,0.300,0.532,0.807,0.459,0.286,0.581,0.408,0.442,0.581,0.579
332,2026-10-21,0.412,0.714,0.530,0.516,0.531,0.282,0.647,0.468,0.411,...,0.300,0.437,0.567,0.487,0.351,0.350,0.350,0.380,0.518,0.595
